In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import json
from datetime import datetime
import time
import random
from tqdm import tqdm
import os

In [2]:
# ========== 1. ĐỌC FILE ĐÃ CHUẨN HÓA ==========
input_file = '2021_CHUANHOA_FIXED.xlsx'  #Thay đổi nếu cần
df = pd.read_excel(input_file)

In [3]:
class RicePriceParser:
    # Bỏ OM 4218 theo yêu cầu
    TRACKED_ITEMS = ['OM 5451', 'OM 18', 'Đài Thơm 8', 'IR 50404', 'Tấm', 'Cám']
    
    KEYWORDS = {
        'OM 5451': [r'OM\s*5451', r'lúa\s*5451'],
        'OM 18': [r'OM\s*18', r'lúa\s*18'],
        'Đài Thơm 8': [r'Đài\s*Thơm\s*8', r'ĐT8'],
        'IR 50404': [r'IR\s*50404', r'504', r'IR\s*504'],
        'Tấm': [r'Tấm'],
        'Cám': [r'Cám']
    }

    @staticmethod
    def extract_price_strictly(keyword, text):
        # Regex cải tiến: Tìm số có dạng X.XXX sau từ khóa, tránh bắt nhầm số hiệu lúa
        pattern = rf"{keyword}(?:[^0-9]*?)(\d{{1,2}}\.?\d{{3}})(?:\s*[-–—]\s*(\d{{1,2}}\.?\d{{3}}))?"
        
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            p1 = match.group(1).replace('.', '')
            
            # Logic chống bắt nhầm số hiệu lúa vào giá
            # Nếu p1 nằm trong chuỗi keyword (ví dụ p1 là 5451) thì bỏ qua
            clean_keyword = re.sub(r'[^0-9]', '', keyword)
            if clean_keyword and p1 == clean_keyword:
                # Tìm tiếp đoạn phía sau xem có giá thật không
                remaining_text = text[match.end():]
                next_match = re.search(r'(\d{1,2}\.\d{3})', remaining_text)
                if next_match:
                    p1 = next_match.group(1).replace('.', '')
                else:
                    return None
            
            # Kiểm tra khoảng giá lúa gạo hợp lệ (4.000 - 25.000)
            if 4000 <= int(p1) <= 25000:
                p2 = match.group(2).replace('.', '') if match.group(2) else None
                return f"{p1}-{p2}" if p2 else p1
        return None

    @classmethod
    def parse_smart_v6(cls, soup):
        results = {item: None for item in cls.TRACKED_ITEMS}
        # Quét theo Table trước vì VietnamBiz thường để giá lúa gạo trong bảng
        tables = soup.find_all('table')
        table_text = " ".join([t.get_text(separator=' ') for t in tables])
        
        # Văn bản tổng hợp
        full_text = soup.get_text(separator=' ')

        for item, aliases in cls.KEYWORDS.items():
            for alias in aliases:
                # Thử tìm trong Table trước, nếu không có mới tìm toàn bài
                price = cls.extract_price_strictly(alias, table_text)
                if not price:
                    price = cls.extract_price_strictly(alias, full_text)
                
                if price:
                    results[item] = price
                    break
        return results

In [5]:
def start_crawl_v6(input_file, output_file):
    # 1. Đọc file
    df = pd.read_excel(input_file)
    url_col, date_col = 'URL', 'Ngày đăng_chuẩn'

    final_output = []
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/122.0.0.0'}

    print(f"🚀 Đang chạy bản V6 - Quét {len(df)} dòng (Đã fix lỗi NaT và Null Tấm/Cám)...")

    for i, row in tqdm(df.iterrows(), total=len(df)):
        url = str(row[url_col]).strip()
        raw_date = row[date_col]
        
        # --- SỬA LỖI NaT TẠI ĐÂY ---
        # Kiểm tra nếu là NaT hoặc ô trống thì để "N/A"
        if pd.isna(raw_date):
            pub_date = "N/A"
        else:
            pub_date = raw_date.strftime('%Y-%m-%d') if hasattr(raw_date, 'strftime') else str(raw_date)

        data_entry = {
            "id": i + 1,
            "pub_date": pub_date,
            "url": url,
            "data": {item: None for item in RicePriceParser.TRACKED_ITEMS},
            "crawl_status": "pending"
        }

        if url.startswith('http') and url.lower() != 'nan':
            try:
                res = requests.get(url, headers=headers, timeout=10)
                if res.status_code == 200:
                    soup = BeautifulSoup(res.content, 'html.parser')
                    # Loại bỏ rác để quét chính xác hơn
                    for s in soup(["script", "style"]): s.decompose()
                    
                    data_entry["data"] = RicePriceParser.parse_smart_v6(soup)
                    data_entry["crawl_status"] = "success"
                else:
                    data_entry["crawl_status"] = f"HTTP {res.status_code}"
            except:
                data_entry["crawl_status"] = "Error"

        final_output.append(data_entry)

    # --- BƯỚC FIX LỖI NULL TẤM/CÁM (Forward & Backward Fill) ---
    # Sau khi cào xong hết mới tiến hành điền giá trị trống
    print("🛠️ Đang xử lý điền giá liên tục cho Tấm và Cám...")
    
    # Fill xuôi: Nếu hôm nay ko có giá, lấy giá ngày hôm trước
    for j in range(1, len(final_output)):
        for item in ['Tấm', 'Cám']:
            if final_output[j]["data"][item] is None:
                final_output[j]["data"][item] = final_output[j-1]["data"][item]
                
    # Fill ngược: Xử lý nốt những ngày đầu tiên nếu bị trống
    for j in range(len(final_output)-2, -1, -1):
        for item in ['Tấm', 'Cám']:
            if final_output[j]["data"][item] is None:
                final_output[j]["data"][item] = final_output[j+1]["data"][item]

    # Lưu file JSON
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(final_output, f, ensure_ascii=False, indent=4)
        
    print(f"\n✅ Hoàn tất! Dữ liệu đã lưu tại: {output_file}")

if __name__ == "__main__":
    start_crawl_v6('2021_chuanhoa.xlsx', 'rice_data_v6_final.json')

🚀 Đang chạy bản V6 - Quét 843 dòng (Đã fix lỗi NaT và Null Tấm/Cám)...


100%|██████████| 843/843 [05:57<00:00,  2.36it/s]

🛠️ Đang xử lý điền giá liên tục cho Tấm và Cám...

✅ Hoàn tất! Dữ liệu đã lưu tại: rice_data_v6_final.json
